# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 1 — Compare Balanced vs Natural Distribution Fine-Tuning (unpaired)

Loads the per-model CSV outputs produced by notebooks `0`–`3` and produces
side-by-side tables with a delta column. Two model directories are loaded:

- `BASE_MODEL` — the no-natural control (e.g. `Qwen/Qwen2.5-14B-Instruct`)
- `FT_MODEL`   — the fine-tuned variant (e.g. `Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_balanced`)

Each side is just `./phase1_metrics/{MODEL_NAME}/` filled by running the four
analysis notebooks twice (once per model). This notebook does no model
inference — it only joins, deltas, and plots.

**Unpaired comparison.** Tables and plots report group-level deltas (per-class
Brier, ECE, ΔH_c, etc.) computed as `natural − balanced` on each side's own
population/per-class point estimates. Because the two runs come from the same
Stage 3 test split they describe the same estimands, but we don't exploit row
alignment here — each side carries its own bootstrap CIs from the upstream
notebooks, and we just put them next to each other.

For a stronger statistical statement (per-row Δp_true CIs, McNemar on argmax
flips, etc.) you'd want a paired notebook that joins the scored parquets on
`row_id`. Out of scope here.

## Section overview

| §  | Output                                                              |
|----|---------------------------------------------------------------------|
| 1  | Setup: knobs + helpers                                              |
| 2  | Tier 1 (Brier / accuracy / κ / macro F1 / weighted F1) — cross-target deltas |
| 3  | Tier 1 — per-target per-class deltas (incl. precision / recall / F1) |
| 4  | Tier 2 (ECE) — cross-target deltas                                  |
| 5  | Tier 3 (ΔH_c flattening) — cross-target and per-class deltas        |
| 6  | PriDe debiasing summary — letter-bias spread, both models           |
| 7  | Headline figure: κ and macro/weighted F1 (Balanced vs Natural Distribution Fine-Tuning)               |
| 8  | Headline figure: per-target Brier (Balanced vs Natural Distribution Fine-Tuning, side by side)        |
| 9  | Headline figure: per-class Brier deltas across all targets          |


## 1. Setup

Set `BASE_MODEL` and `FT_MODEL` to the MODEL_NAME strings you used when running
notebooks `0`–`3`. The viz notebook will read CSVs from
`./phase1_metrics/{name}/` for each side.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- The two models being compared ------------------------------------------
# These must match the MODEL_NAME values you set at the top of notebooks 0-3
# when running on each side.

# MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_raw_05_12_lauren"
# MODEL_NAME = "Qwen2.5-14B-Instruct-FT_balanced_canonical_step07_bf16_test_raw_05_12"


BASE_MODEL = "Qwen2.5-14B-Instruct-FT_balanced_canonical_step07_bf16_test_raw_05_12"

FT_MODEL   = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_raw_05_12_lauren"
# --- Display knobs ----------------------------------------------------------
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# --- I/O --------------------------------------------------------------------
# Same convention as in the analysis notebooks: per-model subdir under
# ./phase1_metrics/, with '/' in HF model IDs replaced by '__'.

def metrics_dir_for(model_name: str) -> Path:
    return Path(
        f"/content/drive/MyDrive/NLP_project/data/"
        f"stage_1_output_{model_name}/phase1_metrics/"
    )

BASE_DIR = metrics_dir_for(BASE_MODEL)
FT_DIR   = metrics_dir_for(FT_MODEL)

COMPARE_DIR = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"compare_metrics__{BASE_MODEL}__vs__{FT_MODEL}/"
)
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE  : {BASE_DIR}  (exists={BASE_DIR.is_dir()})")
print(f"natural    : {FT_DIR}    (exists={FT_DIR.is_dir()})")
print(f"OUT   : {COMPARE_DIR}")

# --- Targets ----------------------------------------------------------------
TARGETS = [
    "SEX",
    "AGE_BINS",
    "EDUCATION_LEVEL",
    "RACE_ETHNICITY",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]

In [ ]:
def load_csv(model_dir: Path, name: str, **read_kwargs) -> pd.DataFrame:
    """Load a CSV from a model's metrics dir; warn (but don't crash) if missing."""
    path = model_dir / name
    if not path.exists():
        warnings.warn(f"Missing {path}")
        return None
    return pd.read_csv(path, **read_kwargs)


def side_by_side(balanced: pd.DataFrame, ft: pd.DataFrame,
                  on: str = None, value_cols: list = None,
                  delta_sign: dict = None) -> pd.DataFrame:
    """Join balanced and natural tables and add an natural-balanced delta column for each value col.

    Args:
      balanced, ft     : DataFrames to compare. Joined on their index unless `on` is given.
      value_cols   : columns to compare. Defaults to numeric columns shared by both.
      delta_sign   : optional dict mapping column name -> +1 or -1 indicating
                     which direction is "improvement" (lower-is-better metrics
                     like Brier get -1, so the delta sign represents
                     "improvement direction"). Used only for the `_improved`
                     boolean column.

    Returns a DataFrame with columns suffixed `_balanced` and `_natural` plus `_delta`.
    """
    if balanced is None or ft is None:
        return None

    if on is not None:
        balanced = balanced.set_index(on) if on in balanced.columns else balanced
        ft   = ft.set_index(on)   if on in ft.columns   else ft

    if value_cols is None:
        # Default: numeric columns present on both sides
        common_num = [c for c in balanced.columns
                      if c in ft.columns
                      and np.issubdtype(balanced[c].dtype, np.number)
                      and np.issubdtype(ft[c].dtype, np.number)]
        value_cols = common_num

    rows = balanced.index.union(ft.index)
    out = pd.DataFrame(index=rows)
    for c in value_cols:
        b = balanced[c].reindex(rows) if c in balanced.columns else pd.Series(np.nan, index=rows)
        f = ft[c].reindex(rows)   if c in ft.columns   else pd.Series(np.nan, index=rows)
        out[f"{c}_balanced"]  = b
        out[f"{c}_natural"]    = f
        out[f"{c}_delta"] = f - b
        if delta_sign and c in delta_sign:
            out[f"{c}_improved"] = (np.sign(out[f"{c}_delta"]) == delta_sign[c])

    return out

## 2. Tier 1: cross-target Brier / accuracy / clipped LL

Loads `cross_target_summary.csv` from each side. This is the headline
population-level table from notebook 1.

`brier_delta` is `natural − balanced`. Lower Brier is better, so a *negative* delta is
improvement; the `brier_improved` column flags this for readability.

`acc_delta` is `natural − balanced`. Higher accuracy is better, so a *positive* delta
is improvement; `acc_improved` flags this.

`skill_brier` is already a "higher = better" skill score so its delta carries
the natural sign.


In [ ]:
tier1_balanced = load_csv(BASE_DIR, "cross_target_summary.csv", index_col="target")
tier1_ft   = load_csv(FT_DIR,   "cross_target_summary.csv", index_col="target")

tier1_compare = side_by_side(
    tier1_balanced, tier1_ft,
    value_cols=[
        "pop_accuracy",
        "baseline_acc",
        "pop_brier",
        "baseline_brier",
        "skill_brier",
        "pop_clipped_ll",
        "kappa",
        "macro_f1",
        "weighted_f1",
    ],
    delta_sign={
        "pop_accuracy":   +1,  # higher is better
        "pop_brier":      -1,  # lower is better
        "skill_brier":    +1,  # higher is better
        "pop_clipped_ll": +1,  # higher (closer to 0) is better
        "kappa":          +1,  # higher is better; 0 = chance, 1 = perfect
        "macro_f1":       +1,  # higher is better
        "weighted_f1":    +1,  # higher is better
    },
)

if tier1_compare is not None:
    print("TIER 1: cross-target population-level comparison")
    print("=" * 80)
    print("\nProbability metrics:")
    cols_to_show = [
        "pop_accuracy_balanced", "pop_accuracy_natural", "pop_accuracy_delta", "pop_accuracy_improved",
        "pop_brier_balanced",    "pop_brier_natural",    "pop_brier_delta",    "pop_brier_improved",
        "skill_brier_balanced",  "skill_brier_natural",  "skill_brier_delta",  "skill_brier_improved",
    ]
    show_cols = [c for c in cols_to_show if c in tier1_compare.columns]
    print(tier1_compare[show_cols].to_string(float_format=lambda x: f"{x:.3f}"))

    # Chance-corrected agreement + F1 family. Reported separately because the
    # column block is wide and these are the headline classification metrics
    # for imbalanced targets.
    cls_cols = [
        "kappa_balanced", "kappa_natural", "kappa_delta", "kappa_improved",
        "macro_f1_balanced", "macro_f1_natural", "macro_f1_delta", "macro_f1_improved",
        "weighted_f1_balanced", "weighted_f1_natural", "weighted_f1_delta", "weighted_f1_improved",
    ]
    cls_show = [c for c in cls_cols if c in tier1_compare.columns]
    if cls_show:
        print("\nClassification metrics (chance-corrected + F1 family):")
        print(tier1_compare[cls_show].to_string(float_format=lambda x: f"{x:.3f}"))

    tier1_compare.to_csv(COMPARE_DIR / "tier1_cross_target_compare.csv")
    print(f"\nSaved -> {COMPARE_DIR / 'tier1_cross_target_compare.csv'}")

## 3. Tier 1: per-target per-class deltas

For each target, load `fidelity_{T}_deb.csv` from both sides and produce a
class-level comparison. This is where the interesting structure lives — the
population summary in §2 hides which classes the natural actually helps vs hurts.

We compare on the debiased probabilities (`_deb`) by default since that's the
canonical reporting kind from notebook 1. Flip `KIND = "raw"` if you want the
non-PriDe comparison.


In [ ]:
KIND = "deb"  # or "raw" for the non-PriDe comparison

per_target_compare = {}
for t in TARGETS:
    b = load_csv(BASE_DIR, f"fidelity_{t}_{KIND}.csv", index_col="class")
    f = load_csv(FT_DIR,   f"fidelity_{t}_{KIND}.csv", index_col="class")
    if b is None or f is None:
        continue
    cmp = side_by_side(
        b, f,
        value_cols=["n_c", "accuracy", "brier", "clipped_ll", "clip_rate",
                    "precision", "recall", "f1"],
        delta_sign={"accuracy": +1, "brier": -1, "clipped_ll": +1,
                    "precision": +1, "recall": +1, "f1": +1},
    )
    per_target_compare[t] = cmp

    print(f"\n[{t}]  ({KIND})")
    print("-" * 72)
    # Probability metrics block
    show_cols = [
        "accuracy_balanced", "accuracy_natural", "accuracy_delta",
        "brier_balanced",    "brier_natural",    "brier_delta",
        "clip_rate_balanced", "clip_rate_natural", "clip_rate_delta",
    ]
    show_cols = [c for c in show_cols if c in cmp.columns]
    if show_cols:
        print("Probability metrics:")
        print(cmp[show_cols].to_string(float_format=lambda x: f"{x:.3f}"))
    # Classification metrics block
    cls_cols = [
        "precision_balanced", "precision_natural", "precision_delta",
        "recall_balanced",    "recall_natural",    "recall_delta",
        "f1_balanced",        "f1_natural",        "f1_delta",
    ]
    cls_show = [c for c in cls_cols if c in cmp.columns]
    if cls_show:
        print("\nClassification metrics (precision / recall / F1):")
        print(cmp[cls_show].to_string(float_format=lambda x: f"{x:.3f}"))

    cmp.to_csv(COMPARE_DIR / f"tier1_fidelity_{t}_{KIND}_compare.csv")

## 4. Tier 2: ECE comparison

Loads `ece_cross_target_summary.csv` from each side and reports deltas on the
four ECE numbers (top-label/classwise × equal-width/equal-mass).

For ECE, *lower is better*, so a negative delta is improvement.


In [ ]:
ece_balanced = load_csv(BASE_DIR, "ece_cross_target_summary.csv", index_col="target")
ece_ft   = load_csv(FT_DIR,   "ece_cross_target_summary.csv", index_col="target")

ece_compare = side_by_side(
    ece_balanced, ece_ft,
    value_cols=["ece_top_label", "ece_classwise", "ece_top_eqm", "ece_cw_eqm"],
    delta_sign={
        "ece_top_label":  -1,
        "ece_classwise":  -1,
        "ece_top_eqm":    -1,
        "ece_cw_eqm":     -1,
    },
)

if ece_compare is not None:
    print("TIER 2: ECE comparison")
    print("=" * 80)
    print("Negative delta = natural is better calibrated than balanced.")
    print()
    show_cols = [
        "ece_top_label_balanced", "ece_top_label_natural", "ece_top_label_delta", "ece_top_label_improved",
        "ece_classwise_balanced", "ece_classwise_natural", "ece_classwise_delta", "ece_classwise_improved",
    ]
    show_cols = [c for c in show_cols if c in ece_compare.columns]
    print(ece_compare[show_cols].to_string(float_format=lambda x: f"{x:.4f}"))
    ece_compare.to_csv(COMPARE_DIR / "tier2_ece_compare.csv")
    print(f"\nSaved -> {COMPARE_DIR / 'tier2_ece_compare.csv'}")

## 5. Tier 3: ΔH_c flattening comparison

Two views:

- **Cross-target**: from `flattening_cross_target_summary.csv`. Positive
  `mean_delta_h` means the LLM is sharper than ACS (under-represents
  within-group heterogeneity — the Wang et al. flattening signal). natural
  *reducing* this number toward zero is improvement, so the natural sign is
  flipped from §2.

- **Per-class**: from `flattening_{T}_{KIND}.csv`. Class-level deltas tell you
  which classes the natural made more heterogeneous (good) vs more flattened (bad).


In [ ]:
# Cross-target flattening summary
flat_balanced = load_csv(BASE_DIR, "flattening_cross_target_summary.csv", index_col="target")
flat_ft   = load_csv(FT_DIR,   "flattening_cross_target_summary.csv", index_col="target")

flat_compare = side_by_side(
    flat_balanced, flat_ft,
    value_cols=["mean_delta_h", "max_delta_h", "min_delta_h",
                "flat_classes", "n_classes_eval", "coverage"],
    # Improvement = natural reduces |mean_delta_h| toward zero. The raw delta sign
    # is informative but ambiguous re: improvement direction; we leave
    # delta_sign empty and report the absolute-value reduction below.
)

if flat_compare is not None:
    # Add an |Δ| reduction column: positive = natural moved mean_delta_h closer to 0
    flat_compare["abs_mean_dh_reduction"] = (
        flat_compare["mean_delta_h_balanced"].abs() - flat_compare["mean_delta_h_natural"].abs()
    )

    print("TIER 3: ΔH_c flattening — cross-target")
    print("=" * 80)
    print("abs_mean_dh_reduction > 0 = natural moved closer to ACS heterogeneity (good)")
    print()
    show_cols = [
        "mean_delta_h_balanced", "mean_delta_h_natural", "mean_delta_h_delta",
        "abs_mean_dh_reduction",
        "flat_classes_balanced", "flat_classes_natural",
    ]
    show_cols = [c for c in show_cols if c in flat_compare.columns]
    print(flat_compare[show_cols].to_string(float_format=lambda x: f"{x:.3f}"))
    flat_compare.to_csv(COMPARE_DIR / "tier3_flattening_cross_target_compare.csv")
    print(f"\nSaved -> {COMPARE_DIR / 'tier3_flattening_cross_target_compare.csv'}")

In [ ]:
# Per-target per-class ΔH_c
per_target_flat_compare = {}
for t in TARGETS:
    b = load_csv(BASE_DIR, f"flattening_{t}_{KIND}.csv", index_col="class")
    f = load_csv(FT_DIR,   f"flattening_{t}_{KIND}.csv", index_col="class")
    if b is None or f is None:
        continue
    cmp = side_by_side(
        b, f,
        value_cols=["n_used", "H_acs", "H_llm", "delta_h"],
    )
    per_target_flat_compare[t] = cmp

    print(f"\n[{t}]  ({KIND})  per-class ΔH_c")
    print("-" * 72)
    show_cols = ["H_llm_balanced", "H_llm_natural", "H_llm_delta",
                 "delta_h_balanced", "delta_h_natural", "delta_h_delta"]
    show_cols = [c for c in show_cols if c in cmp.columns]
    print(cmp[show_cols].to_string(float_format=lambda x: f"{x:.3f}"))
    cmp.to_csv(COMPARE_DIR / f"tier3_flattening_{t}_{KIND}_compare.csv")

## 6. PriDe debiasing summary

Letter-position spread under raw vs debiased probs. We expect both models to
behave similarly here (PriDe is a post-hoc correction on the logits, not a
property of the model), but a large discrepancy would be a flag.


In [ ]:
pride_balanced = load_csv(BASE_DIR, "bias_summary.csv", index_col="target")
pride_ft   = load_csv(FT_DIR,   "bias_summary.csv", index_col="target")

pride_compare = side_by_side(
    pride_balanced, pride_ft,
    value_cols=["mean_spread_raw", "mean_spread_deb",
                "max_spread_raw", "max_spread_deb", "reduction_pct"],
)

if pride_compare is not None:
    print("PriDe: letter-position bias summary")
    print("=" * 80)
    show_cols = [
        "mean_spread_raw_balanced", "mean_spread_raw_natural", "mean_spread_raw_delta",
        "mean_spread_deb_balanced", "mean_spread_deb_natural", "mean_spread_deb_delta",
        "reduction_pct_balanced",   "reduction_pct_natural",
    ]
    show_cols = [c for c in show_cols if c in pride_compare.columns]
    print(pride_compare[show_cols].to_string(float_format=lambda x: f"{x:.3f}"))
    pride_compare.to_csv(COMPARE_DIR / "pride_bias_compare.csv")
    print(f"\nSaved -> {COMPARE_DIR / 'pride_bias_compare.csv'}")

## 7. Headline figure: κ and macro/weighted F1 (Balanced vs Natural Distribution Fine-Tuning)

Single panel per metric. For each metric, one horizontal pair of points per
target (balanced on top, natural below) with bootstrap CI bars. Chance reference (κ=0
or F1 = 1/n_Y) marked as a vertical line.

These are the chance-corrected and class-imbalance-aware analogues of
accuracy — they're the right headline numbers when class frequencies vary
across targets the way they do in this study.


In [ ]:
# Load the cross-target tables again so we can pull the *_lo / *_hi columns
# for CI bars. (They're in the CSV but not retained in tier1_compare.)

def _load_cross_target(model_dir: Path) -> pd.DataFrame:
    return load_csv(model_dir, "cross_target_summary.csv", index_col="target")


base_ct = _load_cross_target(BASE_DIR)
ft_ct   = _load_cross_target(FT_DIR)

if base_ct is None or ft_ct is None:
    print("Missing cross_target_summary.csv on one side; skipping figure.")
else:
    metrics_to_plot = [
        ("kappa",       "Cohen's κ",       0.0),
        ("macro_f1",    "Macro F1",        None),    # baseline = 1/n_Y, per target
        ("weighted_f1", "Weighted F1",     None),
    ]

    fig, axes = plt.subplots(
        nrows=1, ncols=len(metrics_to_plot),
        figsize=(5.0 * len(metrics_to_plot), 0.45 * len(TARGETS) + 2),
        squeeze=False,
    )

    # Targets that exist on both sides
    targets_both = [t for t in TARGETS if t in base_ct.index and t in ft_ct.index]

    for j, (metric, label, chance_line) in enumerate(metrics_to_plot):
        ax = axes[0, j]
        lo_col = f"{metric}_lo"
        hi_col = f"{metric}_hi"

        if metric not in base_ct.columns or metric not in ft_ct.columns:
            ax.set_visible(False)
            continue

        y = np.arange(len(targets_both))
        bar_h = 0.35

        b_pe = base_ct.loc[targets_both, metric].values
        f_pe = ft_ct.loc[targets_both, metric].values

        if lo_col in base_ct.columns and hi_col in base_ct.columns:
            b_lo = base_ct.loc[targets_both, lo_col].values
            b_hi = base_ct.loc[targets_both, hi_col].values
            f_lo = ft_ct.loc[targets_both, lo_col].values
            f_hi = ft_ct.loc[targets_both, hi_col].values
            b_err = np.array([b_pe - b_lo, b_hi - b_pe])
            f_err = np.array([f_pe - f_lo, f_hi - f_pe])
        else:
            b_err = None
            f_err = None

        ax.errorbar(b_pe, y + bar_h/2,
                     xerr=b_err, fmt="o", color="#888", ecolor="#bbb",
                     capsize=3, markersize=5, label="Balanced SFT")
        ax.errorbar(f_pe, y - bar_h/2,
                     xerr=f_err, fmt="o", color="#3a6f9b", ecolor="#7fa6c4",
                     capsize=3, markersize=5, label="Natural SFT")

        # Reference line for chance.
        if chance_line is not None:
            ax.axvline(chance_line, color="#c44e52", linestyle="--",
                       linewidth=1.0, label=f"chance ({chance_line:.2f})")
        else:
            # Macro / weighted F1 chance level = 1/n_Y per target; varies, so
            # we draw a faint marker per target instead.
            for yi, t in enumerate(targets_both):
                n_Y = int(base_ct.loc[t, "n_classes"])
                ax.plot([1/n_Y], [yi], marker="|", color="#c44e52",
                        markersize=14, alpha=0.6,
                        label="chance (1/n_Y)" if yi == 0 else None)

        ax.set_yticks(y)
        ax.set_yticklabels(targets_both, fontsize=9)
        ax.set_xlabel(label)
        ax.set_title(label, fontsize=11)
        ax.legend(loc="upper right", fontsize=8, frameon=True)
        ax.grid(axis="x", linestyle=":", alpha=0.4)
        ax.spines[["top", "right"]].set_visible(False)
        # κ ranges roughly [-1, 1]; F1 in [0, 1]. Set sensible x-lims.
        if metric == "kappa":
            ax.set_xlim(min(-0.05, b_pe.min() - 0.05, f_pe.min() - 0.05),
                        max(1.0, b_pe.max() + 0.05, f_pe.max() + 0.05))
        else:
            ax.set_xlim(0.0, 1.0)

    fig.suptitle("Chance-corrected and F1-family metrics: Balanced vs Natural Distribution Fine-Tuning",
                 fontsize=13, y=1.02)
    plt.tight_layout()
    fig_path = COMPARE_DIR / "fig_compare_kappa_f1.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.savefig(COMPARE_DIR / "fig_compare_kappa_f1.pdf", bbox_inches="tight")
    plt.show()
    print(f"Saved -> {fig_path}")

    #put

## 8. Headline figure: per-target Brier (Balanced vs Natural Distribution Fine-Tuning)

One panel per target. Two horizontal bars per class — balanced on top, natural below —
with their bootstrap CIs from the upstream notebooks. The uniform baseline
is shown as a vertical reference. Classes are ordered by the natural Brier so the
"worst-handled class on natural" reads at the bottom of each panel.


In [ ]:
def _load_per_class_with_ci(model_dir: Path, target: str, kind: str = "deb") -> pd.DataFrame:
    df = load_csv(model_dir, f"fidelity_{target}_{kind}.csv", index_col="class")
    if df is None:
        return None
    # Drop the non-per-class summary rows so plots / deltas operate only on real classes.
    return df.drop(["POPULATION", "UNIFORM_BASELINE",
                    "MACRO_AVG", "WEIGHTED_AVG"], errors="ignore")


targets_with_data = [
    t for t in TARGETS
    if _load_per_class_with_ci(BASE_DIR, t, KIND) is not None
    and _load_per_class_with_ci(FT_DIR, t, KIND) is not None
]

if not targets_with_data:
    print("No per-target fidelity CSVs found on either side; skipping figure.")
else:
    ncols = 2
    nrows = (len(targets_with_data) + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(7 * ncols, 2.5 + 1.3 * max(
            len(_load_per_class_with_ci(BASE_DIR, t, KIND).index)
            for t in targets_with_data
        )),
        squeeze=False,
    )

    for i, t in enumerate(targets_with_data):
        ax = axes[i // ncols, i % ncols]
        b = _load_per_class_with_ci(BASE_DIR, t, KIND)
        f = _load_per_class_with_ci(FT_DIR, t, KIND)

        # Sort classes by natural Brier ascending; align both sides
        classes = f.sort_values("brier", ascending=True).index.tolist()
        b = b.reindex(classes)
        f = f.reindex(classes)

        y = np.arange(len(classes))
        bar_h = 0.35

        def err(df):
            pe = df["brier"].to_numpy()
            lo = df["brier_lo"].to_numpy() if "brier_lo" in df else pe
            hi = df["brier_hi"].to_numpy() if "brier_hi" in df else pe
            return pe, np.array([pe - lo, hi - pe])

        b_pe, b_err = err(b)
        f_pe, f_err = err(f)

        ax.errorbar(b_pe, y + bar_h/2, xerr=b_err, fmt="o", color="#888",
                     ecolor="#bbb", capsize=3, markersize=5, label="Balanced SFT")
        ax.errorbar(f_pe, y - bar_h/2, xerr=f_err, fmt="o", color="#3a6f9b",
                     ecolor="#7fa6c4", capsize=3, markersize=5, label="Natural SFT")

        # Uniform baseline
        n_Y = len(classes)
        baseline = 1.0 - 1.0 / n_Y
        ax.axvline(baseline, color="#c44e52", linestyle="--", linewidth=1.0,
                   label=f"uniform baseline ({baseline:.2f})")

        ax.set_yticks(y)
        ax.set_yticklabels(classes, fontsize=9)
        ax.set_xlabel("Brier score (lower = better)")
        ax.set_title(f"{t}  (n_Y={n_Y}, kind={KIND})", fontsize=11)
        ax.set_xlim(0, max(2.0, baseline * 1.1))
        ax.legend(loc="upper right", fontsize=8, frameon=True)
        ax.grid(axis="x", linestyle=":", alpha=0.4)
        ax.spines[["top", "right"]].set_visible(False)

    # Hide unused panels
    for j in range(len(targets_with_data), nrows * ncols):
        axes[j // ncols, j % ncols].set_visible(False)

    fig.suptitle(f"Per-class Brier: Balanced vs Natural Distribution Fine-Tuning  ({KIND}, 95% bootstrap CIs)",
                 fontsize=13, y=1.00)
    plt.tight_layout()
    fig_path = COMPARE_DIR / "fig_compare_brier_per_class.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.savefig(COMPARE_DIR / "fig_compare_brier_per_class.pdf", bbox_inches="tight")
    plt.show()
    print(f"Saved -> {fig_path}")

## 9. Headline figure: per-class Brier delta across all targets

Single panel: every (target, class) as a row, sorted by Δ Brier. Negative
Δ (left of zero) = natural improved that class; positive Δ (right of zero) = natural
hurt that class. Useful for spotting whether the natural systematically helps
minority classes vs hurts majority ones.


In [ ]:
rows = []
for t in TARGETS:
    b = _load_per_class_with_ci(BASE_DIR, t, KIND)
    f = _load_per_class_with_ci(FT_DIR, t, KIND)
    if b is None or f is None:
        continue
    for c in b.index:
        if c not in f.index:
            continue
        rows.append({
            "target":      t,
            "class":       c,
            "n_c":         int(b.loc[c].get("n_c", 0)) if not pd.isna(b.loc[c].get("n_c", np.nan)) else 0,
            "brier_balanced":  b.loc[c]["brier"],
            "brier_natural":    f.loc[c]["brier"],
            "brier_delta": f.loc[c]["brier"] - b.loc[c]["brier"],
            "acc_balanced":    b.loc[c]["accuracy"],
            "acc_natural":      f.loc[c]["accuracy"],
            "acc_delta":   f.loc[c]["accuracy"] - b.loc[c]["accuracy"],
        })

delta_df = pd.DataFrame(rows)
if len(delta_df) > 0:
    delta_df = delta_df.sort_values("brier_delta", ascending=True)
delta_df.to_csv(COMPARE_DIR / "per_class_delta_table.csv", index=False)

if len(delta_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 0.3 * len(delta_df) + 1))
    labels = [f"{r.target}: {r['class']}  (n={int(r.n_c)})"
              for _, r in delta_df.iterrows()]
    y = np.arange(len(delta_df))
    colors = ["#3a6f9b" if d < 0 else "#c44e52" for d in delta_df["brier_delta"]]
    ax.barh(y, delta_df["brier_delta"].values, color=colors, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Δ Brier (natural − balanced)   ←  natural better   |   balanced better  →")
    ax.set_title(f"Per-class Brier delta: natural vs balanced  ({KIND})", fontsize=11)
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig_path = COMPARE_DIR / "fig_brier_delta_per_class.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.savefig(COMPARE_DIR / "fig_brier_delta_per_class.pdf", bbox_inches="tight")
    plt.show()
    print(f"\nSaved -> {fig_path}")
    print(f"\nTop 5 classes where natural helped most:")
    print(delta_df.head(5).to_string(float_format=lambda x: f"{x:.3f}"))
    print(f"\nTop 5 classes where natural hurt most:")
    print(delta_df.tail(5).to_string(float_format=lambda x: f"{x:.3f}"))

In [ ]:
def plot_brier_delta_clean(
    delta_df,
    compare_dir,
    kind="deb",
    figsize_width=9,
):
    """
    Cleaner subgroup-level Brier delta visualization.

    Negative delta:
        Natural SFT better

    Positive delta:
        Balanced SFT better
    """

    import matplotlib.pyplot as plt
    import numpy as np

    if len(delta_df) == 0:
        print("delta_df is empty.")
        return

    # ============================================================
    # PRETTY TARGET NAMES
    # ============================================================

    pretty_target_names = {
        "SEX": "Sex",
        "AGE_BINS": "Age",
        "RACE_ETHNICITY": "Race/Ethnicity",
        "EDUCATION_LEVEL": "Education",
        "INCTOT_BINS": "Income",
        "HEALTH_INSURANCE": "Health Insurance",
        "ENGLISH_STATUS": "English Status",
    }

    # ============================================================
    # SORT
    # ============================================================

    target_order = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
    ]

    plot_df = delta_df.copy()

    plot_df["target"] = pd.Categorical(
        plot_df["target"],
        categories=target_order,
        ordered=True,
    )

    plot_df = (
        plot_df
        .sort_values(
            ["target", "brier_delta"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
)
    # ============================================================
    # CLEAN LABELS
    # ============================================================

    labels = [
        f"{row['class']}\n({pretty_target_names.get(row.target, row.target)})"
        for _, row in plot_df.iterrows()
    ]

    # ============================================================
    # COLORS
    # ============================================================

    colors = [
        "#3a6f9b" if d < 0 else "#c44e52"
        for d in plot_df["brier_delta"]
    ]

    # ============================================================
    # FIGURE
    # ============================================================

    fig_height = 0.42 * len(plot_df) + 1.5

    fig, ax = plt.subplots(
        figsize=(figsize_width, fig_height)
    )

    y = np.arange(len(plot_df))

    ax.barh(
        y,
        plot_df["brier_delta"].values,
        color=colors,
        edgecolor="white",
        linewidth=0.6,
    )

    # ============================================================
    # REFERENCE LINE
    # ============================================================

    ax.axvline(
        0,
        color="black",
        linewidth=0.9,
    )

    # ============================================================
    # LABELS
    # ============================================================

    ax.set_yticks(y)
    ax.set_yticklabels(
        labels,
        fontsize=8,
    )

    ax.set_xlabel(
        "Δ Brier Score (Natural − Balanced)\n"
        "Negative = Natural SFT Better     "
        "Positive = Balanced SFT Better",
        fontsize=10,
    )

    ax.set_title(
        "Subgroup-Level Calibration Differences "
        "Between Fine-Tuning Strategies",
        fontsize=12,
        pad=10,
    )

    # ============================================================
    # GRID/STYLING
    # ============================================================

    ax.grid(
        axis="x",
        linestyle=":",
        alpha=0.4,
    )

    ax.spines[["top", "right"]].set_visible(False)

    # ============================================================
    # LAYOUT
    # ============================================================

    plt.tight_layout()

    # ============================================================
    # SAVE
    # ============================================================

    png_path = (
        compare_dir /
        f"fig_brier_delta_clean_{kind}.png"
    )

    pdf_path = (
        compare_dir /
        f"fig_brier_delta_clean_{kind}.pdf"
    )

    plt.savefig(
        png_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()

    print(f"\nSaved PNG -> {png_path}")
    print(f"Saved PDF -> {pdf_path}")
plot_brier_delta_clean(
    delta_df,
    COMPARE_DIR,
    kind=KIND,
)

In [ ]:
def make_brier_delta_table(
    delta_df,
    compare_dir,
    kind="deb",
):
    """
    Create a clean subgroup-level Brier delta table
    instead of a graph.
    """

    import pandas as pd

    if len(delta_df) == 0:
        print("delta_df is empty.")
        return

    # ============================================================
    # PRETTY TARGET NAMES
    # ============================================================

    pretty_target_names = {
        "SEX": "Sex",
        "AGE_BINS": "Age",
        "RACE_ETHNICITY": "Race/Ethnicity",
        "EDUCATION_LEVEL": "Education",
        "INCTOT_BINS": "Income",
        "HEALTH_INSURANCE": "Health Insurance",
        "ENGLISH_STATUS": "English Status",
    }

    # ============================================================
    # SORT
    # ============================================================

    target_order = [
        "SEX",
        "AGE_BINS",
        "RACE_ETHNICITY",
        "EDUCATION_LEVEL",
        "INCTOT_BINS",
        "HEALTH_INSURANCE",
        "ENGLISH_STATUS",
    ]

    table_df = delta_df.copy()

    table_df["target"] = pd.Categorical(
        table_df["target"],
        categories=target_order,
        ordered=True,
    )

    table_df = (
        table_df
        .sort_values(
            ["target", "brier_delta"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

    # ============================================================
    # CLEAN FORMATTING
    # ============================================================

    table_df["Target"] = table_df["target"].map(
        pretty_target_names
    )

    table_df["Subgroup"] = table_df["class"]

    table_df["Δ Brier"] = table_df["brier_delta"].round(4)

    table_df["Better Model"] = table_df["brier_delta"].apply(
        lambda x:
            "Natural" if x < 0
            else "Balanced"
    )

    # ============================================================
    # FINAL TABLE
    # ============================================================

    final_table = table_df[
        [
            "Target",
            "Subgroup",
            "Δ Brier",
            "Better Model",
        ]
    ]

    # ============================================================
    # SAVE
    # ============================================================

    csv_path = (
        compare_dir /
        f"table_brier_delta_{kind}.csv"
    )

    latex_path = (
        compare_dir /
        f"table_brier_delta_{kind}.tex"
    )

    final_table.to_csv(
        csv_path,
        index=False,
    )

    # Optional LaTeX export
    with open(latex_path, "w") as f:
        f.write(
            final_table.to_latex(
                index=False,
                escape=False,
            )
        )

    # ============================================================
    # DISPLAY
    # ============================================================

    display(final_table)

    print(f"\nSaved CSV  -> {csv_path}")
    print(f"Saved TEX  -> {latex_path}")

    return final_table


# ============================================================
# RUN
# ============================================================

brier_table = make_brier_delta_table(
    delta_df,
    COMPARE_DIR,
    kind=KIND,
)

In [ ]:
def make_brier_win_summary_table(delta_df):
    """
    Summarize how often each implementation
    achieves better subgroup-level Brier scores.
    """

    import pandas as pd

    # ============================================================
    # OVERALL SUMMARY
    # ============================================================

    total_subgroups = len(delta_df)

    balanced_wins = (delta_df["brier_delta"] > 0).sum()
    natural_wins = (delta_df["brier_delta"] < 0).sum()
    ties = (delta_df["brier_delta"] == 0).sum()

    overall_table = pd.DataFrame({
        "Model Better": [
            "Balanced",
            "Natural",
            "Tie",
        ],

        "Subgroups Won": [
            balanced_wins,
            natural_wins,
            ties,
        ],

        "Percent of Subgroups": [
            round(100 * balanced_wins / total_subgroups, 1),
            round(100 * natural_wins / total_subgroups, 1),
            round(100 * ties / total_subgroups, 1),
        ]
    })

    # ============================================================
    # PER-TARGET SUMMARY
    # ============================================================

    per_target_rows = []

    for target, group in delta_df.groupby("target"):

        total = len(group)

        balanced = (group["brier_delta"] > 0).sum()
        natural = (group["brier_delta"] < 0).sum()
        tie = (group["brier_delta"] == 0).sum()

        per_target_rows.append({
            "Target": target,
            "Balanced Better": balanced,
            "Natural Better": natural,
            "Ties": tie,
            "Balanced %": round(100 * balanced / total, 1),
            "Natural %": round(100 * natural / total, 1),
        })

    per_target_table = pd.DataFrame(per_target_rows)

    # ============================================================
    # DISPLAY
    # ============================================================

    print("\nOVERALL SUMMARY")
    display(overall_table)

    print("\nPER-TARGET SUMMARY")
    display(per_target_table)

    return overall_table, per_target_table


# ============================================================
# RUN
# ============================================================

overall_table, per_target_table = (
    make_brier_win_summary_table(delta_df)
)

In [ ]:
# ============================================================
# MACRO F1 RESULTS
# ============================================================

macro_f1_df = pd.DataFrame({

    "Target Variable": [
        "SEX",
        "AGE_BINS",
        "EDUCATION_LEVEL",
        "RACE_ETHNICITY",
        "INCTOT_BINS",
        "HEALTH_INSURANCE",
        "ENGLISH_STATUS",
    ],

    "Balanced SFT": [
        0.576,
        0.373,
        0.365,
        0.224,
        0.276,
        0.580,
        0.549,
    ],

    "Natural SFT": [
        0.558,
        0.298,
        0.274,
        0.166,
        0.221,
        0.519,
        0.497,
    ],
})

# ============================================================
# DIFFERENCE COLUMN
# ============================================================

macro_f1_df["Difference\n(Balanced - Natural)"] = (
    macro_f1_df["Balanced SFT"]
    - macro_f1_df["Natural SFT"]
).round(3)

# ============================================================
# ROUNDING
# ============================================================

for col in [
    "Balanced SFT",
    "Natural SFT",
]:
    macro_f1_df[col] = (
        macro_f1_df[col].round(3)
    )

# ============================================================
# DISPLAY
# ============================================================

print("\nMacro F1 Comparison\n")

print(
    macro_f1_df.to_string(index=False)
)

# ============================================================
# EXPORT LATEX TABLE
# ============================================================

latex_table = macro_f1_df.to_latex(
    index=False,
    escape=False,
    column_format="lccc",
    caption=(
        "Macro F1 comparison between balanced "
        "and population-weighted natural fine-tuning. "
        "Higher values indicate improved subgroup-balanced "
        "classification performance."
    ),
    label="tab:macrof1_compare",
)

with open(
    "macro_f1_table.tex",
    "w"
) as f:
    f.write(latex_table)

print("\nSaved LaTeX table:")
print("macro_f1_table.tex")

In [ ]:
import pandas as pd

# ============================================================
# OVERALL BRIER SCORE COMPARISON TABLE
# Corrected evaluation outputs
# ============================================================

brier_df = pd.DataFrame({

    "Target Variable": [
        "SEX",
        "AGE_BINS",
        "RACE_ETHNICITY",
        "EDUCATION_LEVEL",
        "INCTOT_BINS",
        "HEALTH_INSURANCE",
        "ENGLISH_STATUS",
    ],

    "Balanced SFT": [
        0.589,
        0.779,
        0.612,
        0.876,
        0.938,
        0.347,
        0.083,
    ],

    "Natural SFT": [
        0.513,
        0.768,
        0.631,
        0.791,
        0.893,
        0.152,
        0.028,
    ],
})

# ============================================================
# DIFFERENCE COLUMN
# Negative values = Natural SFT better
# Positive values = Balanced SFT better
# ============================================================

brier_df["Difference\n(Natural - Balanced)"] = (
    brier_df["Natural SFT"]
    - brier_df["Balanced SFT"]
).round(3)

# ============================================================
# ROUND DISPLAY
# ============================================================

for col in [
    "Balanced SFT",
    "Natural SFT",
]:
    brier_df[col] = (
        brier_df[col].round(3)
    )

# ============================================================
# DISPLAY
# ============================================================

print("\nOverall Brier Score Comparison\n")

print(
    brier_df.to_string(index=False)
)

# ============================================================
# EXPORT LATEX TABLE
# ============================================================

latex_table = brier_df.to_latex(
    index=False,
    escape=False,
    column_format="lccc",
    caption=(
        "Overall Brier score comparison between "
        "balanced and population-weighted natural "
        "fine-tuning. Lower values indicate "
        "improved probabilistic calibration. "
        "Overall Brier scores are aggregated across "
        "all observations, meaning dominant demographic "
        "categories contribute more heavily to the "
        "final metric."
    ),
    label="tab:brier_compare",
)

with open(
    "brier_score_table.tex",
    "w"
) as f:
    f.write(latex_table)

print("\nSaved LaTeX table:")
print("brier_score_table.tex")